# 第18次课：Rasterio栅格数据处理

**地学编程基础** | 2学时（90分钟）

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **理解**栅格数据的本质 = 数组（NumPy）+ 元数据（空间参考信息）；
2. **掌握** Rasterio 库读取 GeoTIFF 文件的基本方法；
3. **能够**读取并查看栅格数据的元数据（profile、transform、CRS）；
4. **能够**读取多波段数据并进行可视化；
5. **能够**结合 NumPy 进行栅格代数运算，独立完成 NDVI 计算；
6. **能够**将处理结果保存为新的 GeoTIFF 文件并保留地理参考信息。

---

## 🛠️ 准备工作

本节课使用的库：`rasterio`、`numpy`、`matplotlib`

如果你的环境中还没有安装 rasterio，请在终端中运行：

```bash
pip install rasterio
```

本节课使用的数据：
- `data/landsat_clip.tif`（多波段Landsat 8/9影像，已裁剪到课程研究区域）

---

## 一、温故知新：从NumPy到Rasterio

### 上节课我们学到了什么？

- DEM（数字高程模型）就是一个二维 NumPy 数组
- 遥感影像的每个波段也是一个二维 NumPy 数组
- 我们可以用 NumPy 对这些数组做各种运算（求平均、求最大值、布尔筛选等）

### 但是有一个问题……

**思考一下**：如果我把一个 DEM 数组用 `np.save()` 保存为 `.npy` 文件，然后发给你，你能在地图上正确显示这块区域吗？

In [ ]:
# 我们先用NumPy模拟一个DEM
import numpy as np

fake_dem = np.random.randint(0, 1000, size=(100, 100))
print(f"数组形状：{fake_dem.shape}")
print(f"数据类型：{fake_dem.dtype}")
print(f"最大高程：{fake_dem.max()} 米")

# 但是这个数组里只有数字，没有：
#   - 这块区域在哪里？（坐标范围）
#   - 用什么坐标系？（投影信息）
#   - 每个像元代表多大面积？（空间分辨率）

### 💡 核心观念

> **栅格数据 = 像元矩阵（数组） + 空间参考（元数据）**

二者缺一不可。这就是今天我们要学的：

- **NumPy** 处理像元矩阵的部分（值、运算）
- **Rasterio** 处理空间参考的部分（坐标、投影、地理变换）

二者结合，才能完整地处理 GeoTIFF 这样的栅格数据格式。

---

## 二、Rasterio入门：读取GeoTIFF

Rasterio 是 Python 中处理栅格数据的事实标准库，由 Mapbox 公司主导开发，底层基于 GDAL。

### 2.1 基本读取模式

Rasterio 使用 **`with` 上下文管理器** 来打开文件，这与我们之前学的文件操作一致：

```python
import rasterio

with rasterio.open('文件路径.tif') as src:
    # 在这个代码块里使用 src 对象
    # 离开代码块时文件会自动关闭
    pass
```

**为什么要用 `with` ？**

栅格文件可能很大，必须及时关闭，否则会占用大量系统资源。`with` 语句保证了**无论代码是否报错，文件都会被正确关闭**。

In [ ]:
# 让我们打开课程数据
import rasterio

with rasterio.open('data/landsat_clip.tif') as src:
    print(f"文件名：{src.name}")
    print(f"文件模式：{src.mode}")
    print(f"是否已关闭：{src.closed}")

# 离开 with 之后，文件就关闭了
print(f"\n离开with后，是否已关闭：{src.closed}")

### 2.2 栅格的元数据

Rasterio 打开文件后，可以通过几个核心属性了解栅格的元数据：

| 属性 | 说明 |
|---|---|
| `src.width` | 影像宽度（列数） |
| `src.height` | 影像高度（行数） |
| `src.count` | 波段数 |
| `src.crs` | 坐标参考系（Coordinate Reference System） |
| `src.transform` | 仿射变换参数（像元坐标 ↔ 地理坐标的转换） |
| `src.bounds` | 影像四至范围（左、下、右、上） |
| `src.dtypes` | 各波段的数据类型 |
| `src.profile` | 上面所有信息的汇总（字典形式） |

In [ ]:
# 查看课程数据的基本元数据
with rasterio.open('data/landsat_clip.tif') as src:
    print(f"影像宽度（列数）：{src.width}")
    print(f"影像高度（行数）：{src.height}")
    print(f"波段数：{src.count}")
    print(f"数据类型：{src.dtypes}")
    print(f"\n坐标参考系（CRS）：\n{src.crs}")
    print(f"\n四至范围：\n{src.bounds}")

In [ ]:
# profile 是一个汇总字典，包含了上面所有关键信息
with rasterio.open('data/landsat_clip.tif') as src:
    profile = src.profile

print("=== 完整的 profile ===")
for key, value in profile.items():
    print(f"  {key}: {value}")

### 2.3 仿射变换：像元坐标 vs. 地理坐标

**思考**：栅格数据存储的时候是按 `[行, 列]` 排列的（像元坐标）。但地理空间是按 `[经度, 纬度]` 或 `[X, Y]` 来定位的（地理坐标）。

二者怎么对应？答案是 **仿射变换矩阵（Affine Transform）**。

`src.transform` 返回一个仿射变换对象，它告诉我们：

- 第 (0, 0) 像元对应的地理坐标是哪里？
- 每移动一个像元，地理坐标变化多少？（即空间分辨率）

In [ ]:
with rasterio.open('data/landsat_clip.tif') as src:
    transform = src.transform
    print(f"仿射变换矩阵：\n{transform}")
    print(f"\n左上角X坐标：{transform.c}")
    print(f"左上角Y坐标：{transform.f}")
    print(f"X方向像元大小：{transform.a}")
    print(f"Y方向像元大小：{transform.e}  (注意：通常为负，因为Y轴向下递增行号)")
    
    # 计算空间分辨率
    print(f"\n空间分辨率：{abs(transform.a)} x {abs(transform.e)} 米/像元")

---

## 🎯 即时练习①

**练习目标**：熟练读取栅格元数据。

**练习题**：打开 `data/landsat_clip.tif`，编写代码完成以下任务：

1. 输出该影像的总像元数（width × height）
2. 输出该影像所采用的坐标参考系的 EPSG 代码
3. 输出该影像覆盖的地理范围面积（单位：平方公里）
4. 判断该影像是否为多波段影像（>1个波段则为多波段）

**提示**：
- EPSG代码可以用 `src.crs.to_epsg()` 获取
- 面积 = 宽度范围 × 高度范围（注意单位换算，米→公里要除以1000）

In [ ]:
# ===== 即时练习① =====
# 在下方编写你的代码：





---

## 三、读取波段数据

前面我们只是看了元数据，还没真正读取像元值。读取波段数据用 `src.read()` 方法。

### 3.1 读取单个波段

```python
src.read(1)   # 读取第1个波段
src.read(4)   # 读取第4个波段（红波段，对Landsat 8/9）
```

**返回值是一个 NumPy 二维数组（H × W）**——这正是我们想要的！现在 NumPy 的所有运算都可以用了。

### ⚠️ 重要：Landsat 波段编号

Landsat 8/9 的常用波段：

| 波段编号 | 名称 | 用途 |
|---|---|---|
| Band 2 | Blue 蓝 | 真彩色合成 |
| Band 3 | Green 绿 | 真彩色合成 |
| **Band 4** | **Red 红** | **真彩色合成 + NDVI 计算** |
| **Band 5** | **NIR 近红外** | **NDVI 计算（关键！）** |
| Band 6 | SWIR1 短波红外 | 水分、地质 |

In [ ]:
# 读取红波段（Band 4）
with rasterio.open('data/landsat_clip.tif') as src:
    red = src.read(4)

print(f"红波段数据类型：{type(red)}")
print(f"数组形状：{red.shape}")
print(f"数据类型：{red.dtype}")
print(f"\n像元值范围：{red.min()} ~ {red.max()}")
print(f"像元值平均值：{red.mean():.2f}")

### 3.2 一次读取多个波段

如果要同时读取多个波段，可以用 `src.read([4, 5, 3])`，返回一个**三维数组**（波段数 × H × W）。

或者直接 `src.read()` （不带参数）读取所有波段。

In [ ]:
# 同时读取红、近红外、绿波段
with rasterio.open('data/landsat_clip.tif') as src:
    bands = src.read([4, 5, 3])  # 红、近红外、绿

print(f"返回数组的形状：{bands.shape}")
print(f"  - 第一维（3）：波段数")
print(f"  - 第二维（{bands.shape[1]}）：行数")
print(f"  - 第三维（{bands.shape[2]}）：列数")

### 3.3 单波段可视化

用 Matplotlib 的 `imshow()` 即可显示。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# 中文显示设置（如果需要）
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 读取红、近红外两个波段
with rasterio.open('data/landsat_clip.tif') as src:
    red = src.read(4)
    nir = src.read(5)

# 并排显示两个波段
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(red, cmap='Reds')
axes[0].set_title('红波段 (Band 4)')
plt.colorbar(im1, ax=axes[0], shrink=0.7)

im2 = axes[1].imshow(nir, cmap='Greens')
axes[1].set_title('近红外波段 (Band 5)')
plt.colorbar(im2, ax=axes[1], shrink=0.7)

plt.tight_layout()
plt.show()

---

## 🎯 即时练习②

**练习目标**：读取多波段并可视化。

**练习题**：基于 `data/landsat_clip.tif`，完成以下任务：

1. 读取蓝波段（Band 2）、绿波段（Band 3）、红波段（Band 4）三个波段；
2. 在一个 1×3 的子图布局中，分别用对应的颜色映射（蓝/绿/红）显示这三个波段；
3. 每个子图加上中文标题。

**提示**：
- Matplotlib 的 cmap 可以使用 `'Blues'`、`'Greens'`、`'Reds'`
- 子图布局：`fig, axes = plt.subplots(1, 3, figsize=(15, 5))`

In [ ]:
# ===== 即时练习② =====
# 在下方编写你的代码：





---

## 四、综合实战：NDVI计算

### 4.1 什么是 NDVI？

**NDVI（Normalized Difference Vegetation Index，归一化植被指数）** 是遥感领域最经典、应用最广的植被指数之一。

**计算公式：**

$$
NDVI = \frac{NIR - Red}{NIR + Red}
$$

**取值范围：[-1, 1]**

| NDVI 值 | 地物含义 |
|---|---|
| < 0 | 水体、雪、云 |
| 0 ~ 0.2 | 裸土、岩石、建成区 |
| 0.2 ~ 0.4 | 稀疏植被、草地 |
| 0.4 ~ 0.7 | 中等植被、农田 |
| > 0.7 | 茂密森林 |

**原理**：健康的植被叶绿素吸收红光（红波段反射率低），而叶肉细胞强烈反射近红外光（近红外波段反射率高），因此植被的 NDVI 值高。

### 4.2 计算流程

1. 读取红波段和近红外波段；
2. 转换数据类型为浮点数（避免整数除法的精度损失）；
3. 套用 NDVI 公式（NumPy 广播机制让我们对整个数组运算）；
4. 处理可能的除零情况；
5. 可视化 + 保存为 GeoTIFF。

In [ ]:
# Step 1: 读取红波段和近红外波段，并转换为浮点数
with rasterio.open('data/landsat_clip.tif') as src:
    red = src.read(4).astype('float32')
    nir = src.read(5).astype('float32')
    profile = src.profile  # 保存元数据，后面写文件要用

print(f"红波段数据范围：{red.min():.2f} ~ {red.max():.2f}")
print(f"近红外波段数据范围：{nir.min():.2f} ~ {nir.max():.2f}")

In [ ]:
# Step 2: 计算 NDVI
# 注意：分母可能为 0，需要处理

# 方法一：用 np.where 处理除零
denominator = nir + red
ndvi = np.where(denominator == 0, 0, (nir - red) / denominator)

# 也可以用 np.errstate 直接忽略警告（但更推荐上面的写法）
# with np.errstate(divide='ignore', invalid='ignore'):
#     ndvi = (nir - red) / (nir + red)

print(f"NDVI 数据范围：{ndvi.min():.3f} ~ {ndvi.max():.3f}")
print(f"NDVI 平均值：{ndvi.mean():.3f}")
print(f"NDVI 数组形状：{ndvi.shape}")

In [ ]:
# Step 3: 可视化 NDVI
fig, ax = plt.subplots(figsize=(10, 8))

# 使用 RdYlGn 配色（红-黄-绿，符合NDVI的语义：红色=低值=非植被，绿色=高值=植被）
im = ax.imshow(ndvi, cmap='RdYlGn', vmin=-0.3, vmax=0.8)
ax.set_title('NDVI（归一化植被指数）', fontsize=14)
ax.axis('off')

cbar = plt.colorbar(im, ax=ax, shrink=0.7)
cbar.set_label('NDVI 值')

plt.tight_layout()
plt.show()

### 4.3 NDVI分级与统计

我们可以对 NDVI 进行分级（结合前面学过的 NumPy 布尔索引），统计研究区内不同类型地物的占比。

In [ ]:
# 统计各类地物占比
total_pixels = ndvi.size

water = np.sum(ndvi < 0)
bare = np.sum((ndvi >= 0) & (ndvi < 0.2))
sparse = np.sum((ndvi >= 0.2) & (ndvi < 0.4))
moderate = np.sum((ndvi >= 0.4) & (ndvi < 0.7))
dense = np.sum(ndvi >= 0.7)

print("=== 研究区地物分布 ===")
print(f"水体/云雪 (NDVI<0)        : {water/total_pixels*100:6.2f}%")
print(f"裸土/建成区 (0~0.2)        : {bare/total_pixels*100:6.2f}%")
print(f"稀疏植被 (0.2~0.4)         : {sparse/total_pixels*100:6.2f}%")
print(f"中等植被 (0.4~0.7)         : {moderate/total_pixels*100:6.2f}%")
print(f"茂密森林 (NDVI>=0.7)       : {dense/total_pixels*100:6.2f}%")

### 4.4 保存 NDVI 为 GeoTIFF

前面我们计算出的 NDVI 还只是一个 NumPy 数组，没有空间参考信息。要保存为可以在 ArcGIS、QGIS 中打开的 GeoTIFF，需要带上 **元数据**。

**关键步骤**：
1. 复制原始影像的 profile（保留 CRS、transform）
2. 修改其中的 `dtype`（NDVI 是浮点数）和 `count`（只有 1 个波段）
3. 用 `rasterio.open()` 以 `'w'` 模式打开新文件，写入 NDVI 数组

In [ ]:
# 准备输出 profile：基于原始 profile 修改
output_profile = profile.copy()
output_profile.update({
    'dtype': 'float32',  # NDVI是浮点数
    'count': 1,          # 只有一个波段
    'compress': 'lzw'    # 启用压缩，减小文件大小
})

print("输出文件的 profile：")
for key, value in output_profile.items():
    print(f"  {key}: {value}")

In [ ]:
# 写入文件
import os
os.makedirs('output', exist_ok=True)  # 确保输出目录存在

output_path = 'output/ndvi_result.tif'

with rasterio.open(output_path, 'w', **output_profile) as dst:
    dst.write(ndvi.astype('float32'), 1)  # 写入第1个波段

print(f"✅ NDVI 已保存到：{output_path}")
print(f"   文件大小：{os.path.getsize(output_path) / 1024:.1f} KB")

In [ ]:
# 验证：重新读取我们刚才写入的文件
with rasterio.open(output_path) as src:
    print(f"读取回来的 NDVI 形状：{src.read(1).shape}")
    print(f"CRS 信息：{src.crs}")
    print(f"四至范围：{src.bounds}")
    print("\n✅ 元数据完整保留，文件可以在 QGIS / ArcGIS 中打开！")

---

## 🎯 即时练习③（综合实战延伸）

**练习目标**：综合运用所学，完成一个完整的栅格分析流程。

**练习题**：基于刚才计算的 NDVI 数据，完成以下任务：

1. 创建一个**植被掩膜**：将 NDVI ≥ 0.4 的像元标记为 1（植被），其他像元标记为 0（非植被）；
2. 统计植被像元的总数，并计算研究区的植被覆盖率（百分比）；
3. 假设每个像元代表 30×30 平方米（Landsat 标准分辨率），计算研究区植被覆盖的总面积（单位：平方公里）；
4. 将植被掩膜保存为新的 GeoTIFF 文件 `output/vegetation_mask.tif`（数据类型用 `uint8`，不浪费存储空间）。

**提示**：
- 植被掩膜可以用 `(ndvi >= 0.4).astype('uint8')` 一行生成
- 写入新 GeoTIFF 时记得修改 profile 中的 `dtype` 为 `'uint8'`

In [ ]:
# ===== 即时练习③ =====
# 在下方编写你的代码：





---

## 五、本节课小结

### 知识点回顾

| 知识点 | 关键API/语法 |
|---|---|
| 打开栅格文件 | `with rasterio.open(path) as src:` |
| 读取元数据 | `src.width / src.height / src.count / src.crs / src.transform / src.profile` |
| 读取单波段 | `src.read(波段编号)` → NumPy 二维数组 |
| 读取多波段 | `src.read([4, 5, 3])` → NumPy 三维数组 |
| 栅格代数 | NumPy 数组运算（广播机制） |
| 写入栅格 | `with rasterio.open(path, 'w', **profile) as dst: dst.write(arr, 1)` |

### 核心思维方式

> **栅格数据 = 数组 + 元数据**

- 数组部分用 NumPy 处理（运算）
- 元数据部分用 Rasterio 处理（空间参考）
- 写出文件时**必须保留元数据**，否则结果就成了"无主数据"

### 课后作业

1. **基础题**：基于本节课数据，计算 **NDWI（归一化水体指数）**，公式为：$NDWI = \frac{Green - NIR}{Green + NIR}$（绿波段是 Band 3，近红外是 Band 5），并保存为 GeoTIFF。

2. **进阶题**：将研究区的 NDVI 按以下5个等级进行分类，生成一张**整数类型**的分级栅格图（数据类型用 `uint8`）：
   - 1：水体（NDVI < 0）
   - 2：裸土/建成区（0 ≤ NDVI < 0.2）
   - 3：稀疏植被（0.2 ≤ NDVI < 0.4）
   - 4：中等植被（0.4 ≤ NDVI < 0.7）
   - 5：茂密森林（NDVI ≥ 0.7）

   保存为 GeoTIFF，并用 Matplotlib 显示分类结果。

3. **思考题**：如果我们想计算 NDVI 的**多年变化**（比如对比 2020 年和 2024 年同一地区的 NDVI 差异），需要做哪些额外步骤？写一段100字以内的处理流程描述。

---

## 📚 延伸阅读

- Rasterio 官方文档：https://rasterio.readthedocs.io/
- 参考教材：《Python地理空间分析指南（第2版）》第6章、第8章8.1节
- 在线资源：USGS EarthExplorer 下载 Landsat 数据：https://earthexplorer.usgs.gov/

---

下节课预告：**第19次课 地图可视化** —— 学习如何用 Matplotlib 和 GeoPandas 制作出版级专题地图。

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**
>
> 编程能力的成长来自于自己一行一行写代码、一次一次调试错误的过程，直接看答案学不到东西。

In [ ]:
# ===== 即时练习① 参考答案 =====
import rasterio

with rasterio.open('data/landsat_clip.tif') as src:
    # 1. 总像元数
    total_pixels = src.width * src.height
    print(f"1. 总像元数：{total_pixels}")
    
    # 2. EPSG 代码
    epsg = src.crs.to_epsg()
    print(f"2. EPSG 代码：{epsg}")
    
    # 3. 覆盖面积（平方公里）
    bounds = src.bounds
    width_m = bounds.right - bounds.left      # 宽度（米）
    height_m = bounds.top - bounds.bottom     # 高度（米）
    area_km2 = (width_m * height_m) / 1_000_000  # 平方米转平方公里
    print(f"3. 覆盖面积：{area_km2:.2f} 平方公里")
    
    # 4. 是否多波段
    is_multiband = src.count > 1
    print(f"4. 是否多波段：{is_multiband}（共 {src.count} 个波段）")

In [ ]:
# ===== 即时练习② 参考答案 =====
import rasterio
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 读取蓝、绿、红三个波段
with rasterio.open('data/landsat_clip.tif') as src:
    blue = src.read(2)
    green = src.read(3)
    red = src.read(4)

# 创建 1×3 子图
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 蓝波段
im1 = axes[0].imshow(blue, cmap='Blues')
axes[0].set_title('蓝波段 (Band 2)')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], shrink=0.7)

# 绿波段
im2 = axes[1].imshow(green, cmap='Greens')
axes[1].set_title('绿波段 (Band 3)')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], shrink=0.7)

# 红波段
im3 = axes[2].imshow(red, cmap='Reds')
axes[2].set_title('红波段 (Band 4)')
axes[2].axis('off')
plt.colorbar(im3, ax=axes[2], shrink=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ===== 即时练习③ 参考答案 =====
import os
import numpy as np
import rasterio

# 1. 创建植被掩膜
vegetation_mask = (ndvi >= 0.4).astype('uint8')

# 2. 统计植被像元数与覆盖率
veg_pixels = np.sum(vegetation_mask == 1)
total_pixels = vegetation_mask.size
coverage_rate = veg_pixels / total_pixels * 100

print(f"植被像元总数：{veg_pixels}")
print(f"研究区植被覆盖率：{coverage_rate:.2f}%")

# 3. 计算植被覆盖面积（每像元 30×30 = 900 平方米）
veg_area_km2 = veg_pixels * 30 * 30 / 1_000_000
print(f"植被覆盖面积：{veg_area_km2:.2f} 平方公里")

# 4. 保存为 GeoTIFF
os.makedirs('output', exist_ok=True)

mask_profile = profile.copy()
mask_profile.update({
    'dtype': 'uint8',
    'count': 1,
    'compress': 'lzw',
    'nodata': 255  # 可选：设置 nodata 值
})

output_path = 'output/vegetation_mask.tif'
with rasterio.open(output_path, 'w', **mask_profile) as dst:
    dst.write(vegetation_mask, 1)

print(f"\n✅ 植被掩膜已保存到：{output_path}")

# 可视化掩膜
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(vegetation_mask, cmap='Greens')
ax.set_title(f'植被掩膜 (NDVI ≥ 0.4)，覆盖率 {coverage_rate:.2f}%')
ax.axis('off')
plt.tight_layout()
plt.show()

---

## 课后作业参考答案

**仅供参考，建议先自己完成。**

In [ ]:
# ===== 课后作业 1：NDWI 计算 =====
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open('data/landsat_clip.tif') as src:
    green = src.read(3).astype('float32')
    nir = src.read(5).astype('float32')
    profile = src.profile

# 计算 NDWI（处理除零）
denominator = green + nir
ndwi = np.where(denominator == 0, 0, (green - nir) / denominator)

print(f"NDWI 范围：{ndwi.min():.3f} ~ {ndwi.max():.3f}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(ndwi, cmap='RdBu', vmin=-0.5, vmax=0.5)
ax.set_title('NDWI（归一化水体指数）')
ax.axis('off')
plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

# 保存
os.makedirs('output', exist_ok=True)
ndwi_profile = profile.copy()
ndwi_profile.update({'dtype': 'float32', 'count': 1, 'compress': 'lzw'})

with rasterio.open('output/ndwi_result.tif', 'w', **ndwi_profile) as dst:
    dst.write(ndwi.astype('float32'), 1)

print("✅ NDWI 已保存")

In [ ]:
# ===== 课后作业 2：NDVI 五级分类 =====
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# 用前面计算好的 ndvi 数组
ndvi_class = np.zeros_like(ndvi, dtype='uint8')
ndvi_class[ndvi < 0]                              = 1   # 水体
ndvi_class[(ndvi >= 0)   & (ndvi < 0.2)]          = 2   # 裸土/建成区
ndvi_class[(ndvi >= 0.2) & (ndvi < 0.4)]          = 3   # 稀疏植被
ndvi_class[(ndvi >= 0.4) & (ndvi < 0.7)]          = 4   # 中等植被
ndvi_class[ndvi >= 0.7]                           = 5   # 茂密森林

# 保存
class_profile = profile.copy()
class_profile.update({'dtype': 'uint8', 'count': 1, 'compress': 'lzw'})

with rasterio.open('output/ndvi_classified.tif', 'w', **class_profile) as dst:
    dst.write(ndvi_class, 1)

# 可视化
colors = ['#1f77b4', '#d2b48c', '#cce5cc', '#66cd66', '#006400']
labels = ['水体', '裸土/建成区', '稀疏植被', '中等植被', '茂密森林']
cmap = ListedColormap(colors)
norm = BoundaryNorm([0.5, 1.5, 2.5, 3.5, 4.5, 5.5], cmap.N)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(ndvi_class, cmap=cmap, norm=norm)
ax.set_title('NDVI 五级分类结果')
ax.axis('off')

cbar = plt.colorbar(im, ax=ax, shrink=0.7, ticks=[1, 2, 3, 4, 5])
cbar.ax.set_yticklabels(labels)

plt.tight_layout()
plt.show()

print("✅ 分类结果已保存")

### 课后作业 3：思考题参考答案

**多时相 NDVI 变化分析的关键步骤：**

1. **空间配准**：确保两期影像覆盖同一区域、采用相同坐标参考系（如必要，使用 `rasterio.warp` 进行重投影）；
2. **几何对齐**：检查两期影像的像元网格是否对齐，必要时进行重采样；
3. **辐射校正**：消除不同时相的大气、光照差异（理想情况下使用地表反射率产品而非DN值）；
4. **季节匹配**：尽量选择相近的物候期（如同样是夏季），避免季节差异造成误判；
5. **差值计算**：`ndvi_diff = ndvi_2024 - ndvi_2020`，正值表示植被改善，负值表示退化；
6. **变化解释**：结合人类活动数据（如建设用地扩张）解读变化原因。